In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

Mounted at /content/drive


In [ ]:
# Step 2: Move to a folder in your Google Drive
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:
!ls /content/drive/MyDrive/550-images-dataset

data_config.yaml  images  SPLIT-DATA	yolo11n.pt
DATA-SPLIT.ipynb  labels  SPLIT-DATA-2


In [ ]:
# Step 3: Clone the Ultralytics repository into Google Drive
!git clone https://github.com/ultralytics/ultralytics.git

Cloning into 'ultralytics'...
remote: Enumerating objects: 91638, done.
remote: Counting objects: 100% (744/744), done.
remote: Compressing objects: 100% (301/301), done.
remote: Total 91638 (delta 620), reused 457 (delta 443), pack-reused 90894 (from 3)
Receiving objects: 100% (91638/91638), 49.25 MiB | 12.17 MiB/s, done.
Resolving deltas: 100% (69165/69165), done.
Updating files: 100% (879/879), done.


SPLIT TEST

In [ ]:
%cd /content/drive/MyDrive/550-images-dataset

/content/drive/MyDrive/550-images-dataset


In [ ]:
import os
import random
import shutil
from pathlib import Path

# Set your dataset path
dataset_dir = Path("/content/drive/MyDrive/550-images-dataset")
image_dir = dataset_dir / "images/train"
label_dir = dataset_dir / "labels/train"

# Output dirs
output_base = Path("/content/drive/MyDrive/550-images-dataset/SPLIT-DATA")
train_image_dir = output_base / "train" / "images"
train_label_dir = output_base / "train" / "labels"
val_image_dir = output_base / "val" / "images"
val_label_dir = output_base / "val" / "labels"

# Create dirs
for d in [train_image_dir, train_label_dir, val_image_dir, val_label_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Get list of image files
image_files = list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.jpeg"))


# Shuffle and split
random.seed(42)  # reproducibility
random.shuffle(image_files)
split_idx = int(0.8 * len(image_files))
train_files = image_files[:split_idx]
val_files = image_files[split_idx:]

# Helper function to copy images and labels
def copy_data(file_list, img_dest, lbl_dest):
    for img_path in file_list:
        lbl_path = label_dir / (img_path.stem + ".txt")
        if lbl_path.exists():  # check if label exists
            shutil.copy2(img_path, img_dest / img_path.name)
            shutil.copy2(lbl_path, lbl_dest / lbl_path.name)

# Copy data
copy_data(train_files, train_image_dir, train_label_dir)
copy_data(val_files, val_image_dir, val_label_dir)

print(f"Training images: {len(train_files)}, Validation images: {len(val_files)}")


Training images: 440, Validation images: 110


DATA SPLIT BY CLASS

In [ ]:
import os
import random
import shutil
from pathlib import Path

# Set your dataset path
dataset_dir = Path("/content/drive/MyDrive/550-images-dataset")
image_dir = dataset_dir / "images/train"
label_dir = dataset_dir / "labels/train"

# Output dirs
output_base = Path("/content/drive/MyDrive/550-images-dataset/SPLIT-DATA-2")
train_image_dir = output_base / "train" / "images"
train_label_dir = output_base / "train" / "labels"
val_image_dir = output_base / "val" / "images"
val_label_dir = output_base / "val" / "labels"

# Create dirs
for d in [train_image_dir, train_label_dir, val_image_dir, val_label_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Get list of image files
image_files = list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.jpeg"))


# Shuffle and split
random.seed(42)  # reproducibility
random.shuffle(image_files)
split_idx = int(0.8 * len(image_files))
train_files = image_files[:split_idx]
val_files = image_files[split_idx:]

# Helper function to copy images and labels
def copy_data(file_list, img_dest, lbl_dest):
    for img_path in file_list:
        lbl_path = label_dir / (img_path.stem + ".txt")
        if lbl_path.exists():  # check if label exists
            shutil.copy2(img_path, img_dest / img_path.name)
            shutil.copy2(lbl_path, lbl_dest / lbl_path.name)

# Copy data
copy_data(train_files, train_image_dir, train_label_dir)
copy_data(val_files, val_image_dir, val_label_dir)

print(f"Training images: {len(train_files)}, Validation images: {len(val_files)}")


Training images: 440, Validation images: 110


Baseline model

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.5 MB/s eta 0:00:00


In [ ]:
import torch
import numpy
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4


In [ ]:
from ultralytics import YOLO

modified_model = YOLO("/content/drive/MyDrive/ultralytics/ultralytics/cfg/models/11/yolo11.yaml")

modified_model.train(
    data    = "/content/drive/MyDrive/550-images-dataset/data_config.yaml",
    epochs  = 400,
    imgsz   = 768,
    batch   = 16,
    seed    = 0,
    device  = "cuda:0",
    optimizer    = "SGD",
    lr0          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    cos_lr       = True,
    conf    = 0.25,
    iou     = 0.45,
    max_det = 500,
    fliplr    = 0.5,
    flipud    = 0.0,
    degrees   = 10.0,
    translate = 0.1,
    scale     = 0.2,
    shear     = 2.0,
    hsv_h     = 0.01,
    hsv_s     = 0.1,
    hsv_v     = 0.1,
    mosaic    = 0.4,
    copy_paste = 0.0,
    project = "/content/drive/MyDrive/550-images-dataset/Results",
    name    = "01-default-yolo11"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ no model scale passed. Assuming scale='n'.
Ultralytics 8.4.28 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.25, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/550-images-dataset/data_config.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=400, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fracti

In [ ]:
!pip install ultralytics
from ultralytics import YOLO